# A1 · A KNN in numpy: vectorized distances, voting and weights

**Type**: Implement from scratch · **Level**: ●●○ independent · **Time**: about 45 minutes · **core path ★**

## Objective

Implement a KNN classifier without loops over the examples: distance matrix (with the identity $\lVert a-b\rVert^2 = \lVert a\rVert^2+\lVert b\rVert^2-2a\cdot b$),
selection of the `k` neighbors, uniform voting and voting weighted by the inverse of the distance, and handling of zero distance. Check that the predictions match `KNeighborsClassifier`.

## Dataset

`load_wine` (178 wines, 13 chemical measurements, 3 classes), split 70/30 and standardized with the scaler learned on the training set: `Z_tr, y_tr, Z_te, y_te`.

## Educational purpose

See that KNN is **all `predict`**: a distance matrix, an `argpartition` and a vote. You will find the two details that make a difference (rounding errors in the distances and distance 0) and how
scikit-learn resolves ties.

## Task

1. `distances(A, B)`: `(len(A), len(B))` matrix of Euclidean distances **without loops**, with the identity above (careful: because of rounding errors the square can come out negative: use `np.maximum(..., 0)` before the root).
2. `nearest(D, k)`: for every row of `D` the indices of the `k` smallest values, sorted by increasing distance (use `np.argpartition` and then sort only the `k` found).
3. `predict_knn(A, X, y, k, weights)`: prediction for every row of `A` with training set `(X, y)`. `weights="uniform"`: class with the most votes (on a tie, the class with the **smallest label**); `weights="distance"`: every neighbor is worth `1/distance`; if a neighbor has distance 0 **only** the neighbors at distance 0 count.
4. Compare with `KNeighborsClassifier` for `k` in `(1, 5, 15)` and for both weights: save in `agreement` a dictionary `(k, weights) -> fraction of equal predictions` on the test set. Then check the limit case: `predict_knn(Z_tr, Z_tr, y_tr, 3, "distance")` must match `y_tr`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_wine
from sklearn.metrics import pairwise_distances
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

X, y = load_wine(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
scaler = StandardScaler().fit(X_tr)
Z_tr, Z_te = scaler.transform(X_tr), scaler.transform(X_te)
print(Z_tr.shape, Z_te.shape, "| classes:", np.bincount(y_tr))

## Your work

In [ ]:
def distances(A, B):
    ...


def nearest(D, k):
    ...

In [ ]:
def predict_knn(A, X, y, k, weights="uniform"):
    ...

In [ ]:
agreement = {}

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`distances`: `sq = (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None, :] - 2 * A @ B.T; np.sqrt(np.maximum(sq, 0))`. `nearest`: `idx = np.argpartition(D, k - 1, axis=1)[:, :k]; order = np.argsort(np.take_along_axis(D, idx, 1), axis=1); np.take_along_axis(idx, order, 1)`. Vote: for every row `np.bincount(y[idx], weights=w, minlength=n_classes).argmax()` (the `argmax` of a vector with equal maxima takes the first, that is the smallest class).'):
    check(nearest(np.array([[3.0, 1.0, 2.0, 5.0]]), 2).tolist() == [[1, 2]], "nearest returns the k smallest indices, in increasing order", "argpartition then sort only the k found")
    D = distances(Z_te, Z_tr)
    check(D.shape == (len(Z_te), len(Z_tr)) and nearest(D, 5).shape == (len(Z_te), 5), "correct shapes", "distances: (len(A), len(B))")
    check(nearest(np.array([[0.0, 0.0, 0.0]]), 1).shape == (1, 1), "nearest also works with equal distances", "mind the ties")
    check(nearest(D, 5).tolist() == np.argsort(D, axis=1)[:, :5].tolist(), "the indices of the neighbors match a full sort", "argpartition(D, k - 1)[:, :k] then sort by distance")
    check(np.allclose(D, pairwise_distances(Z_te, Z_tr), atol=1e-6) and not np.isnan(distances(Z_tr, Z_tr)).any(),
          "distances equal to pairwise_distances, with no NaN even between identical points", "np.maximum(..., 0) before the root")
    for (k, weights), fraction in agreement.items():
        print(f"   k={k:2d} weights={weights:8s} agreement with scikit-learn: {fraction:.3f}")
    check(set(agreement) == {(k, p) for k in (1, 5, 15) for p in ("uniform", "distance")} and all(v == 1.0 for v in agreement.values()), "all the predictions match scikit-learn", "check the ties (smallest label) and the weights 1/d")
    check((predict_knn(Z_tr, Z_tr, y_tr, 3, "distance") == y_tr).all(), "with 1/distance weights every training point predicts itself (distance 0)", "if there is a neighbor at distance 0 only those at distance 0 count")
done()

---
The commented solution is in `solutions/A_from_scratch/A1_knn_classifier_from_scratch.ipynb`. Open it after you pass the check, or after you have really tried.